# Exploración de datos — Premier League

Este notebook carga partidos desde SQLite (`../data/premier_league.db`), realiza limpieza y transformaciones con **pandas** y genera gráficos con **matplotlib** y **seaborn**.

> Si la base aún no existe, ejecuta antes `01_Extraccion_Datos.ipynb` (o el script de extracción) para poblar `data/premier_league.db`.

## 1. Importación de librerías

In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Estilo reproducible para gráficos
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100

## 2. Conexión a SQLite y listado de tablas

Ruta relativa al notebook en `notebooks/` → carpeta raíz del proyecto `../data/premier_league.db`.

In [ ]:
DB_PATH = Path("../data/premier_league.db").resolve()

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró la base de datos en {DB_PATH}. "
        "Genera primero los datos con el notebook de extracción."
    )

conn = sqlite3.connect(DB_PATH)

tablas = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn,
)
tablas

### Carga de la tabla principal `partidos`

In [ ]:
schema = pd.read_sql_query("PRAGMA table_info(partidos);", conn)
schema

In [ ]:
query = """
SELECT id, fecha, equipo_local, equipo_visitante,
       goles_local, goles_visitante
FROM partidos
"""

df_raw = pd.read_sql_query(query, conn)
conn.close()

df_raw.head(10)

## 3. Limpieza y calidad de datos

- Tipos coherentes (`fecha` como fecha-hora).
- Comprobación de duplicados por `id`.
- Detección y tratamiento de goles ausentes (**NaN**) o negativos (si los hubiera).

In [ ]:
print("Dimensiones:", df_raw.shape)
print("Duplicados id:", df_raw["id"].duplicated().sum())
df_raw.info()

In [ ]:
df = df_raw.copy()

df["fecha"] = pd.to_datetime(df["fecha"], utc=True, errors="coerce")
df.rename(columns={"fecha": "fecha_utc"}, inplace=True)

for col in ("goles_local", "goles_visitante"):
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

nulos_goles = df[["goles_local", "goles_visitante"]].isna().any(axis=1).sum()
print("Filas con al menos un gol NaN antes de eliminar:", nulos_goles)

df = df.dropna(subset=["fecha_utc", "goles_local", "goles_visitante"])

negativos = ((df["goles_local"] < 0) | (df["goles_visitante"] < 0)).sum()
print("Marcadores negativos (deberían ser 0):", negativos)
df = df[(df["goles_local"] >= 0) & (df["goles_visitante"] >= 0)]

df = df.sort_values("fecha_utc").reset_index(drop=True)
df.info()

## 4. Variables derivadas para el análisis

In [ ]:
df["total_goles"] = df["goles_local"] + df["goles_visitante"]


def resultado(fila):
    gl, gv = fila["goles_local"], fila["goles_visitante"]
    if gl > gv:
        return "Victoria_local"
    if gv > gl:
        return "Victoria_visitante"
    return "Empate"


df["resultado"] = df.apply(resultado, axis=1)
df["mes"] = df["fecha_utc"].dt.to_period("M").astype(str)
dias_es = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
df["dia_semana"] = df["fecha_utc"].dt.weekday.map(lambda i: dias_es[i])

df.describe(include="all").transpose().head(20)

## 5. Resumen estadístico

In [ ]:
resumen_numericos = df[["goles_local", "goles_visitante", "total_goles"]].describe()
resumen_numericos

In [ ]:
freq_resultado = (
    df["resultado"].value_counts(normalize=True).rename(" proporcion")
    .to_frame()
)
freq_resultado["conteo"] = df["resultado"].value_counts()
freq_resultado

## 6. Visualizaciones

### 6.1 Distribución de goles por partido (local vs visitante)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["goles_local"], bins=range(0, int(df["goles_local"].max()) + 2), ax=ax[0])
ax[0].set_title("Goles equipo local")
ax[0].set_xlabel("Goles")

sns.histplot(df["goles_visitante"], bins=range(0, int(df["goles_visitante"].max()) + 2), ax=ax[1])
ax[1].set_title("Goles equipo visitante")
ax[1].set_xlabel("Goles")

plt.tight_layout()
plt.show()

### 6.2 Total de goles por encuentro

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df["total_goles"], kde=True)
plt.xlabel("Total de goles")
plt.ylabel("Partidos")
plt.title("Distribución del total de goles por partido")
plt.show()

### 6.3 Proporción de resultados (local / visitante / empate)

In [ ]:
orden_resultado = ["Victoria_local", "Empate", "Victoria_visitante"]
conteos = df["resultado"].value_counts().reindex(orden_resultado).fillna(0)

plt.figure(figsize=(8, 4))
ax = sns.barplot(x=conteos.index, y=conteos.values, palette="Set2")
ax.set_xlabel("Resultado")
ax.set_ylabel("Número de partidos")
ax.set_title("Frecuencia de resultados")
plt.show()

### 6.4 Equipos con más goles a favor en el dataset cargado

Se cuentan goles marcados como local **y** como visitante por nombre de equipo.

In [ ]:
locales = df.groupby("equipo_local")["goles_local"].sum().rename_axis("equipo")
visitantes = df.groupby("equipo_visitante")["goles_visitante"].sum().rename_axis("equipo")

goles_favor = locales.add(visitantes, fill_value=0).sort_values(ascending=False)
top_n = goles_favor.head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_n.values, y=top_n.index, palette="flare")
plt.xlabel("Goles a favor (suma)")
plt.ylabel("Equipo")
plt.title("Top 15 equipos — goles a favor")
plt.tight_layout()
plt.show()

### 6.5 Evolución temporal: promedio móvil de total de goles

Útil para ver tendencias suaves a lo largo de la temporada (ventana centrada).

In [ ]:
serie = df.set_index("fecha_utc")["total_goles"].sort_index()
ventana = min(9, len(serie))
if ventana >= 3:
    media_movil = serie.rolling(window=ventana, center=True).mean()
else:
    media_movil = serie

plt.figure(figsize=(12, 4))
serie.plot(alpha=0.25, label="Total por partido")
media_movil.plot(linewidth=2, label=f"Media móvil ({ventana} partidos)")
plt.legend()
plt.title("Total de goles por partido y tendencia")
plt.ylabel("Goles")
plt.xlabel("Fecha")
plt.tight_layout()
plt.show()

### 6.6 Matriz de correlación (variables numéricas)

In [ ]:
num_cols = df.select_dtypes(include=["number"]).drop(columns=["id"], errors="ignore")
corr = num_cols.corr(numeric_only=True)

plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1)
plt.title("Correlación entre variables numéricas")
plt.tight_layout()
plt.show()

## 7. Tabla lista para exportar / modelado posterior

In [ ]:
df.tail()

## Conclusiones

- Se conectó a SQLite con `sqlite3` y se cargó la tabla `partidos` en un `DataFrame` de pandas.
- La limpieza incluyó conversión de fechas a UTC, validación de goles y eliminación de filas incompletas.
- Las variables derivadas (`total_goles`, `resultado`, agregados temporales) facilitan resúmenes y gráficos.
- Los gráficos muestran distribución de goles, frecuencia de resultados, equipos más ofensivos y una tendencia temporal del ritmo de goles en la muestra disponible.